# Laboratorio 2 · Extraer, comprobar e iterar

**Versión de trabajo para revisión docente.** Usa ejemplos sintéticos de la clase y arranca en modo simulado. El dominio y la muestra definitivos llegan tras la votación.

**Objetivo:** pedirle a un modelo que llene un formulario a partir de documentos, comparar dos instrucciones (sin ejemplos y con ejemplos) y medir cuántos campos acierta cada una, cuánto tarda y cuántos tokens usa.

Trabajen en el equipo del proyecto y alternen quién ejecuta y quién revisa. El producto del equipo vive en `labs/lab02/`; la reflexión y el historial del asistente van al portafolio individual.

## 1. Preparar el entorno

`MODE` indica cómo se ejecuta el notebook:

- `simulated`: usa respuestas preparadas, sin llamar a ningún modelo. Sirve para recorrer el notebook completo sin clave.
- `real`: consulta el modelo configurado. Para usarlo, escribe `CURSO_MODE=real` y tu clave en el archivo `.env` (o en los secretos de Codespaces o Colab), y reinicia el notebook.

Nunca escribas la clave en el notebook: quedaría guardada en el repositorio y en el historial.

El modelo no aparece en el código: se elige por configuración con el rol `generation`.

In [ ]:
import json
import os
from pathlib import Path
from typing import Literal

import pandas as pd
from pydantic import BaseModel, Field
from pydantic_ai import Agent

from curso_ia import (
    create_model, document_input, load_config, load_dataset, load_json,
    save_experiment, show_document, status, timed_run,
)

MODE = os.environ.get("CURSO_MODE", "simulated")
os.environ["CURSO_MODE"] = MODE
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
config = load_config(ROOT)
model_ref = config.models["generation"]
print("Modo:", config.mode, "· Modelo:", model_ref)
display(pd.DataFrame(status(config)))

## 2. Leer los documentos y la referencia humana

La muestra tiene seis mensajes de clientes y una imagen con el mismo contenido que el segundo mensaje. La **referencia** es la respuesta correcta que anotó una persona: con ella mediremos al modelo. El modelo nunca la ve.

Lee cada documento y revisa que la referencia esté bien antes de consultar el modelo. Deja `REFERENCE_REVIEWED = False` hasta hacerlo y anota cualquier desacuerdo.

Dos reglas de la anotación: si la persona pide reemplazo y reembolso a la vez sin decidir, la acción queda vacía (`None`); y un monto que no aparece queda vacío, pero un monto de 0 escrito en el mensaje se conserva como 0.

In [ ]:
# Con la muestra definitiva, indica aquí la ruta a su manifiesto.
DATASET_PATH = None  # Ejemplo: Path("data/lab02/cases.json")
dataset = load_dataset(2, DATASET_PATH)
cases = dataset["cases"]
REFERENCE_REVIEWED = False

print("Conjunto:", dataset["name"], "· Documentos:", len(cases))
pd.DataFrame([{"case_id": case["id"], **case["reference"], "note": case["annotation_note"]}
              for case in cases])

In [ ]:
for case in cases:
    print("─────", case["id"], "·", case["document"])
    show_document(case["path"])

## 3. Definir el formulario que debe llenar el modelo

El **esquema** es el formulario: dice qué campos debe devolver el modelo y de qué tipo es cada uno. PydanticAI se lo entrega al modelo y revisa que la respuesta lo cumpla; si no lo cumple, le pide que lo corrija una vez.

Cada campo tiene tres partes:

- `value`: el dato extraído. `str | None` significa «texto o vacío»; `int | None`, «número entero o vacío». `Literal[...]` limita las respuestas a las opciones indicadas.
- `confidence`: qué tan seguro *dice* estar el modelo, de 0 a 1. Es una declaración, no una garantía; más adelante la comparamos con los aciertos.
- `evidence`: el fragmento del documento que respalda el valor, o vacío si no hay respaldo.

Las descripciones (`description`) también las lee el modelo: son parte de la instrucción.

In [ ]:
class OrderId(BaseModel):
    value: str | None = Field(description="Código del pedido tal como aparece; null si no aparece")
    confidence: float = Field(ge=0, le=1, description="Qué tan seguro dice estar el modelo, de 0 a 1")
    evidence: str | None = Field(description="Fragmento copiado del documento que respalda el valor")


class RequestedAction(BaseModel):
    value: Literal["replacement", "refund"] | None = Field(
        description="Qué pide la persona; null si no lo dice o pide ambas cosas sin decidir")
    confidence: float = Field(ge=0, le=1, description="Qué tan seguro dice estar el modelo, de 0 a 1")
    evidence: str | None = Field(description="Fragmento copiado del documento que respalda el valor")


class Amount(BaseModel):
    value: int | None = Field(description="Monto en pesos, como número entero; null si no aparece")
    confidence: float = Field(ge=0, le=1, description="Qué tan seguro dice estar el modelo, de 0 a 1")
    evidence: str | None = Field(description="Fragmento copiado del documento que respalda el valor")


class Extraction(BaseModel):
    order_id: OrderId
    requested_action: RequestedAction
    amount: Amount


FIELDS = list(Extraction.model_fields)
print("Campos:", FIELDS)

## 4. Escribir dos instrucciones: sin ejemplos y con ejemplos

Las dos versiones comparten esquema, reglas, modelo y documentos. La única diferencia es que la segunda (*few-shot*) agrega dos ejemplos resueltos al final. Así, si cambian los resultados, sabemos a qué atribuirlo.

Los ejemplos no son documentos de la muestra: si lo fueran, el modelo vería la respuesta antes de ser evaluado.

In [ ]:
BASE_INSTRUCTION = """
Extrae order_id, requested_action y amount del documento.
Usa solo lo que el documento dice. No inventes datos.
Acción: replacement, refund o null. Monto: pesos enteros o null.
Usa null si el dato falta o es ambiguo; si piden reemplazo y reembolso sin decidir, la acción es null.
No reemplaces un monto desconocido por 0 ni un 0 escrito por null.
En evidence copia un fragmento breve del documento; si no hay respaldo, usa null.
""".strip()

examples = "\n\n".join(
    f"Mensaje: {example['text']}\nRespuesta: {json.dumps(example['reference'], ensure_ascii=False)}"
    for example in dataset["prompt_examples"]
)

INSTRUCTIONS = {
    "baseline": BASE_INSTRUCTION,
    "few_shot": BASE_INSTRUCTION + "\n\nEjemplos resueltos:\n\n" + examples,
}
SETTINGS = {"temperature": 0.0, "max_tokens": 1200}
print(INSTRUCTIONS["few_shot"])

## 5. Ejecutar con PydanticAI

Para cada documento se crea un `Agent` con el modelo, el esquema (`output_type`) y la instrucción, y se le envía el documento. La imagen se envía como imagen, no como su texto.

`timed_run` hace la consulta y anota la respuesta, el tiempo y los tokens. Si una consulta falla, queda registrada como error y cuenta como fallo en las métricas; el resto sigue.

Las dos versiones se alternan documento por documento (sin ejemplos, con ejemplos, siguiente documento…). Si ejecutáramos primero todas las de una versión, cualquier lentitud pasajera del servicio la perjudicaría solo a ella.

En modo simulado las respuestas están preparadas y no se miden tiempos ni tokens.

In [ ]:
rows = []
for case in cases:
    document = document_input(case["path"])
    for variant, instruction in INSTRUCTIONS.items():
        # En modo simulado se usa la respuesta preparada; en modo real se ignora.
        prepared = case.get("prepared_outputs", {}).get(variant)
        model = create_model(model_ref, config, simulated_output=prepared)
        agent = Agent(model, output_type=Extraction, instructions=instruction,
                      model_settings=SETTINGS, retries=1)
        row = await timed_run(agent, ["Extrae los campos del documento adjunto.", document],
                              ref=model_ref, config=config)
        rows.append({"variant": variant, "case_id": case["id"], **row})
        print(case["id"], variant, row["status"], row.get("error_type", ""))

## 6. Guardar la evidencia

El archivo guarda las instrucciones, el esquema, la configuración (sin claves), las referencias y todas las respuestas. Con él, otra persona puede recalcular las métricas sin llamar al modelo.

Cada ejecución sobrescribe el archivo. Si quieres conservar una ejecución anterior, haz commit antes de volver a ejecutar.

In [ ]:
evidence_path = ROOT / "labs/lab02/evidence/experiment.json"
payload = {
    "dataset": dataset["name"],
    "models": {"generation": model_ref},
    "settings": SETTINGS,
    "output_schema": Extraction.model_json_schema(),
    "instructions": INSTRUCTIONS,
    "cases": [{key: case[key] for key in ("id", "document", "reference", "annotation_note")}
              for case in cases],
    "reference_reviewed": REFERENCE_REVIEWED,
    "results": rows,
    "decisions": {"change": "", "evidence": "", "remaining_error": ""},
}
save_experiment(evidence_path, payload, config)
print("Evidencia guardada en", evidence_path)

## 7. Medir aciertos por campo

Desde aquí se trabaja solo con el archivo guardado.

- **Exactitud**: cuántas veces el valor coincide con la referencia, dividido por todos los documentos. Un vacío frente a una referencia vacía cuenta como acierto. Una consulta fallida cuenta como error.
- **Completitud**: cuántas veces el modelo llenó el campo. Llenar más no es mejor si lo llenado es inventado.

La tabla de errores sirve para ir al documento original y entender qué pasó. La métrica no revisa si la cita de `evidence` es fiel: eso se mira a mano.

In [ ]:
saved = load_json(evidence_path)
reference = {case["id"]: case["reference"] for case in saved["cases"]}

details = []
for row in saved["results"]:
    ok = row["status"] == "ok"
    for field in FIELDS:
        extracted = row["output"][field] if ok else {}
        predicted = extracted.get("value")
        expected = reference[row["case_id"]][field]
        details.append({
            "variant": row["variant"], "case_id": row["case_id"], "field": field,
            "expected": expected, "predicted": predicted,
            "correct": ok and predicted == expected,
            "completed": ok and predicted is not None,
            "failed": not ok,
            "confidence": extracted.get("confidence"),
            "evidence": extracted.get("evidence"),
        })
detail = pd.DataFrame(details)

metrics = detail.groupby(["variant", "field"]).agg(
    correct=("correct", "sum"), cases=("correct", "size"),
    completed=("completed", "sum"), failed=("failed", "sum"),
)
metrics["accuracy"] = metrics["correct"] / metrics["cases"]
metrics["completeness"] = metrics["completed"] / metrics["cases"]
display(metrics)

print("Errores para revisar en el original:")
display(detail[~detail["correct"]][["variant", "case_id", "field", "expected", "predicted", "failed"]])

## 8. Comparar tiempos y tokens

Agregar ejemplos alarga la instrucción: el modelo recibe más **tokens de entrada** (los fragmentos de texto en que se divide lo que lee) en cada consulta. Eso tiene un costo, que medimos junto con el tiempo de respuesta.

El tiempo depende también de la red, de la carga del servicio y de los reintentos automáticos cuando el nivel gratuito limita las solicitudes por minuto. Por eso usamos la **mediana** (el valor del medio, que no se deja arrastrar por una consulta muy lenta) y miramos la tabla por documento. Con siete documentos y una sola ejecución, una diferencia pequeña no es concluyente.

In [ ]:
runs = pd.DataFrame(saved["results"])
if saved["mode"] == "simulated":
    print("El modo simulado no mide tiempos ni tokens: ejecuta en modo real para esta comparación.")
else:
    runs = runs[runs["status"] == "ok"].copy()
    runs["latency_s"] = runs["latency_ms"] / 1000
    timing = runs.groupby("variant").agg(
        median_latency_s=("latency_s", "median"),
        max_latency_s=("latency_s", "max"),
        mean_input_tokens=("input_tokens", "mean"),
        mean_output_tokens=("output_tokens", "mean"),
        total_cost_usd=("estimated_cost_usd", "sum"),
    )
    display(timing.round(2))
    display(runs.pivot(index="case_id", columns="variant", values="latency_s").round(1))

## 9. Contrastar la confianza y explicar el cambio

La primera tabla toma los campos en que el modelo dijo tener al menos 0,8 de confianza y cuenta cuántos acertó. Si declara mucha confianza y se equivoca seguido, no conviene usar esa cifra para decidir sin revisión humana. En modo simulado, las confianzas son inventadas.

Después:

1. Elige un error, búscalo en el documento original y explica por qué ocurrió.
2. Cambia **una sola** parte de la instrucción para corregirlo.
3. Haz commit de la evidencia actual y vuelve a ejecutar desde la sección 5. Revisa también si el arreglo rompió algo que antes funcionaba.
4. Completa `DECISIONS` con lo que cambiaste, la evidencia que lo justifica (aciertos, tiempos o tokens) y un error que sigue sin resolverse.

In [ ]:
HIGH_CONFIDENCE = 0.8
confident = detail[detail["confidence"] >= HIGH_CONFIDENCE]
if confident.empty:
    print("Ningún campo supera la confianza elegida.")
else:
    confidence_check = confident.groupby("variant").agg(
        fields=("correct", "size"), correct=("correct", "sum"))
    confidence_check["accuracy"] = confidence_check["correct"] / confidence_check["fields"]
    display(confidence_check)

DECISIONS = {
    "change": "",           # Qué cambiaste en la instrucción
    "evidence": "",         # Qué resultado lo justifica
    "remaining_error": "",  # Un error que sigue y por qué
}
saved["decisions"] = DECISIONS
saved["reference_reviewed"] = REFERENCE_REVIEWED
save_experiment(evidence_path, saved, config)

## 10. Qué se entrega

**Repositorio del equipo:** este notebook, los documentos de la muestra, `labs/lab02/evidence/experiment.json` y la explicación de un cambio que mejoró algo (o de uno que no mejoró). Solo lo validado pasa después a `src/`.

**Portafolio de cada integrante:** reflexión, historial completo del asistente que usaste (o la declaración de que no usaste uno), sin datos sensibles ni claves, y el enlace al producto del equipo. La pregunta individual de cinco minutos se responde donde indique el docente.

**Antes de presentarlo como evidencia:** ejecuta en modo real con la muestra definitiva, revisa la referencia y completa las decisiones. Una ejecución simulada solo comprueba que el notebook funciona. Si el proveedor falla, registra el bloqueo en vez de reemplazarlo por una simulación.

Para profundizar (en inglés, unos 10 minutos cada una, revisadas el 20/10/2026): [PydanticAI: salida estructurada](https://pydantic.dev/docs/ai/core-concepts/output/) y [entrada multimodal](https://pydantic.dev/docs/ai/core-concepts/input/).